# 🤖 AI Resume Screening System with LangChain & LangSmith Tracing

**Assignment:** GenAI – AI Resume Screening System with Tracing  
**Organization:** Innomatics Technology Hub  
**Internship:** Data Science Internship – February 2026

---

## 📌 Pipeline Architecture
```
Resume → Skill Extraction → Matching → Scoring → Explanation → LangSmith Tracing
```

## 📁 Project Structure
```
ai_resume_screening/
├── prompts/
│   ├── extraction_prompt.py
│   ├── matching_prompt.py
│   └── scoring_prompt.py
├── chains/
│   ├── extraction_chain.py
│   ├── matching_chain.py
│   └── scoring_chain.py
├── main.py
└── AI_Resume_Screening_System.ipynb  ← (this file)
```

## 📦 Step 0: Install Dependencies

In [ ]:
# Install required packages
!pip install langchain langchain-openai langchain-community langsmith openai python-dotenv -q
print("✅ All packages installed successfully!")

## 🔑 Step 1: Environment Setup & LangSmith Configuration

> ⚠️ **Replace the placeholder values below with your actual API keys before running.**  
> Get your LangSmith key at: https://smith.langchain.com  
> Get your OpenAI key at: https://platform.openai.com

In [ ]:
import os

# ─────────────────────────────────────────────
# 🔑 SET YOUR API KEYS HERE
# ─────────────────────────────────────────────
os.environ["OPENAI_API_KEY"]            = "sk-...your-openai-key..."       # <-- Replace
os.environ["LANGCHAIN_API_KEY"]         = "ls__...your-langsmith-key..."   # <-- Replace
os.environ["LANGCHAIN_TRACING_V2"]      = "true"          # Enables LangSmith tracing
os.environ["LANGCHAIN_PROJECT"]         = "AI-Resume-Screening"           # Project name in LangSmith
os.environ["LANGCHAIN_ENDPOINT"]        = "https://api.smith.langchain.com"

print("✅ Environment variables configured.")
print(f"📡 LangSmith Tracing: {os.environ['LANGCHAIN_TRACING_V2']}")
print(f"📂 LangSmith Project : {os.environ['LANGCHAIN_PROJECT']}")

## 📥 Step 2: Import Libraries

In [ ]:
import json
import re
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langsmith import Client

print("✅ All libraries imported successfully!")

## 📄 Step 3: Define Sample Resumes & Job Description
Three candidate profiles: **Strong**, **Average**, and **Weak**

In [ ]:
# ═══════════════════════════════════════════
# 📋 JOB DESCRIPTION – Data Scientist Role
# ═══════════════════════════════════════════
job_description = """
Job Title: Data Scientist

We are looking for an experienced Data Scientist to join our AI team.

Required Skills:
- Python (Pandas, NumPy, Scikit-learn)
- Machine Learning (Supervised & Unsupervised)
- Deep Learning (TensorFlow or PyTorch)
- SQL and data wrangling
- Data visualization (Matplotlib, Seaborn, Tableau)
- NLP experience
- MLOps / model deployment
- Strong communication and presentation skills

Experience Required: 3+ years in a data science or ML role

Preferred:
- Experience with cloud platforms (AWS, GCP, Azure)
- Knowledge of LLMs or GenAI
- Publications or open-source contributions
"""

# ═══════════════════════════════════════════
# 👤 CANDIDATE 1: STRONG
# ═══════════════════════════════════════════
resume_strong = """
Name: Aryan Sharma
Experience: 5 years

Summary:
Senior Data Scientist with 5 years of experience building end-to-end ML pipelines.

Technical Skills:
- Python (Pandas, NumPy, Scikit-learn, XGBoost)
- Deep Learning: TensorFlow, PyTorch
- NLP: spaCy, HuggingFace Transformers, LLMs
- SQL, PostgreSQL, MongoDB
- Data Visualization: Matplotlib, Seaborn, Tableau
- MLOps: Docker, MLflow, Airflow
- Cloud: AWS SageMaker, GCP BigQuery
- LangChain, OpenAI API, RAG systems

Work Experience:
- Senior Data Scientist @ TechCorp (3 years): Led ML model deployment on AWS, reduced churn by 22%.
- Data Scientist @ StartupAI (2 years): Built NLP pipelines using HuggingFace and fine-tuned LLMs.

Education: B.Tech Computer Science, IIT Delhi
Achievements: Published 2 ML papers; contributed to open-source scikit-learn.
"""

# ═══════════════════════════════════════════
# 👤 CANDIDATE 2: AVERAGE
# ═══════════════════════════════════════════
resume_average = """
Name: Priya Mehta
Experience: 2.5 years

Summary:
Data Analyst transitioning into Data Science with foundational ML knowledge.

Technical Skills:
- Python (Pandas, NumPy, basic Scikit-learn)
- Machine Learning: Linear/Logistic Regression, Decision Trees
- SQL, Excel
- Data Visualization: Matplotlib, Power BI

Work Experience:
- Data Analyst @ Analytics Ltd (2.5 years): Built dashboards, ran A/B tests, basic predictive models.

Education: B.Sc Statistics, Delhi University
Certifications: Coursera ML Specialization (Andrew Ng)
"""

# ═══════════════════════════════════════════
# 👤 CANDIDATE 3: WEAK
# ═══════════════════════════════════════════
resume_weak = """
Name: Rahul Gupta
Experience: 6 months

Summary:
Recent graduate interested in data and technology.

Technical Skills:
- Basic Python scripting
- MS Excel, Word, PowerPoint
- Basic HTML

Work Experience:
- Intern @ LocalShop (6 months): Data entry and report formatting.

Education: B.Com, Mumbai University
Certifications: None
"""

# All resumes packaged with labels
candidates = [
    {"name": "Aryan Sharma",  "label": "Strong",  "resume": resume_strong},
    {"name": "Priya Mehta",   "label": "Average", "resume": resume_average},
    {"name": "Rahul Gupta",   "label": "Weak",    "resume": resume_weak},
]

print(f"✅ Loaded {len(candidates)} candidate resumes and 1 job description.")

## 🧩 Step 4: Define Prompts (`prompts/` module)

In [ ]:
# ═══════════════════════════════════════════════════════════
# prompts/extraction_prompt.py
# Purpose: Extract structured info from the resume
# ═══════════════════════════════════════════════════════════
extraction_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
You are an expert HR assistant. Extract the following information from the resume below.
IMPORTANT: Only extract information that is explicitly stated in the resume. Do NOT assume or infer skills not mentioned.

Extract:
1. Skills (technical tools, programming languages, frameworks)
2. Years of Experience
3. Job Titles / Roles held
4. Education background

Output as a clean JSON with keys: "skills", "experience_years", "roles", "education".
Only output valid JSON. No extra text.

Resume:
{resume}
"""
)

# ═══════════════════════════════════════════════════════════
# prompts/matching_prompt.py
# Purpose: Match extracted skills against the job description
# ═══════════════════════════════════════════════════════════
matching_prompt = PromptTemplate(
    input_variables=["extracted_info", "job_description"],
    template="""
You are an expert recruiter. Compare the candidate's extracted profile with the job description.
IMPORTANT: Only evaluate based on what is present in the extracted profile. Do NOT assume any missing skills.

Candidate Profile:
{extracted_info}

Job Description:
{job_description}

Output a JSON with:
- "matched_skills": list of skills the candidate HAS that match job requirements
- "missing_skills": list of required skills the candidate is MISSING
- "experience_match": true/false (does experience meet the requirement?)
- "match_summary": 1-2 sentence summary of how well this candidate fits

Only output valid JSON. No extra text.
"""
)

# ═══════════════════════════════════════════════════════════
# prompts/scoring_prompt.py
# Purpose: Assign a fit score (0-100) with explanation
# ═══════════════════════════════════════════════════════════
scoring_prompt = PromptTemplate(
    input_variables=["match_result", "job_description"],
    template="""
You are a senior hiring manager. Based on the match analysis below, assign a Fit Score from 0 to 100.

Scoring Guide:
- 80-100: Excellent fit, meets almost all requirements
- 60-79:  Good fit, meets most requirements with some gaps
- 40-59:  Average fit, meets some requirements but notable gaps
- 20-39:  Weak fit, meets very few requirements
- 0-19:   Poor fit, does not meet the requirements

Match Analysis:
{match_result}

Job Description (for reference):
{job_description}

Output a JSON with:
- "fit_score": integer between 0 and 100
- "grade": one of ["Excellent", "Good", "Average", "Weak", "Poor"]
- "explanation": 3-4 sentences explaining exactly WHY this score was assigned
- "recommendation": one of ["Strongly Recommend", "Recommend", "Consider", "Do Not Recommend"]

Only output valid JSON. No extra text.
"""
)

print("✅ Prompts defined: extraction_prompt, matching_prompt, scoring_prompt")

## ⛓️ Step 5: Build LangChain Chains (`chains/` module)
Using **LCEL (LangChain Expression Language)** and `.invoke()` method

In [ ]:
# Initialize the LLM (ChatOpenAI)
llm = ChatOpenAI(
    model="gpt-3.5-turbo",   # Use gpt-4 for better results if available
    temperature=0,           # Deterministic output for consistent scoring
)

output_parser = StrOutputParser()

# ═══════════════════════════════════════════════════════════
# chains/extraction_chain.py
# LCEL Chain: PromptTemplate | LLM | OutputParser
# ═══════════════════════════════════════════════════════════
extraction_chain = extraction_prompt | llm | output_parser

# ═══════════════════════════════════════════════════════════
# chains/matching_chain.py
# ═══════════════════════════════════════════════════════════
matching_chain = matching_prompt | llm | output_parser

# ═══════════════════════════════════════════════════════════
# chains/scoring_chain.py
# ═══════════════════════════════════════════════════════════
scoring_chain = scoring_prompt | llm | output_parser

print("✅ LCEL Chains built: extraction_chain, matching_chain, scoring_chain")
print("   Each chain uses: PromptTemplate | ChatOpenAI | StrOutputParser")

## ⚙️ Step 6: Full Pipeline Function (`main.py` logic)
Combines all 4 steps: Extract → Match → Score → Explain

In [ ]:
def safe_parse_json(text: str) -> dict:
    """
    Safely parse JSON from LLM output.
    Strips markdown code fences if present.
    """
    try:
        # Strip ```json ... ``` blocks if any
        cleaned = re.sub(r"```json|```", "", text).strip()
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        print(f"⚠️  JSON parse error: {e}")
        return {"raw_output": text}


def run_screening_pipeline(resume: str, job_description: str, candidate_name: str, label: str) -> dict:
    """
    main.py – Full AI Resume Screening Pipeline
    
    Steps:
    1. Skill Extraction  – extract skills, experience, roles from resume
    2. Matching Logic    – compare candidate profile with job requirements
    3. Scoring           – assign a 0-100 fit score
    4. Explanation       – provide reasoning for the score
    
    LangSmith traces all 3 chain invocations automatically.
    """
    print(f"\n{'='*60}")
    print(f"🔍 Processing: {candidate_name} ({label} Candidate)")
    print(f"{'='*60}")

    # ── STEP 1: Skill Extraction ──────────────────────────────
    print("  ⚙️  Step 1: Extracting skills from resume...")
    extraction_output = extraction_chain.invoke({"resume": resume})
    extracted_info    = safe_parse_json(extraction_output)
    print(f"  ✅ Extracted: {list(extracted_info.keys())}")

    # ── STEP 2: Matching Logic ────────────────────────────────
    print("  ⚙️  Step 2: Matching with job description...")
    matching_output = matching_chain.invoke({
        "extracted_info":  json.dumps(extracted_info, indent=2),
        "job_description": job_description
    })
    match_result = safe_parse_json(matching_output)
    print(f"  ✅ Matched: {len(match_result.get('matched_skills', []))} skills | "
          f"Missing: {len(match_result.get('missing_skills', []))} skills")

    # ── STEP 3 & 4: Scoring + Explanation ────────────────────
    print("  ⚙️  Step 3 & 4: Scoring and generating explanation...")
    scoring_output = scoring_chain.invoke({
        "match_result":    json.dumps(match_result, indent=2),
        "job_description": job_description
    })
    score_result = safe_parse_json(scoring_output)
    print(f"  ✅ Score: {score_result.get('fit_score')}/100 | Grade: {score_result.get('grade')}")

    # ── Combine all results ───────────────────────────────────
    final_result = {
        "candidate_name":  candidate_name,
        "label":           label,
        "extracted_info":  extracted_info,
        "match_result":    match_result,
        "score_result":    score_result,
    }
    return final_result


print("✅ Pipeline function defined: run_screening_pipeline()")

## 🚀 Step 7: Run the Pipeline on All 3 Candidates
LangSmith will automatically trace all runs.

In [ ]:
# Run pipeline for all 3 candidates
all_results = []

for candidate in candidates:
    result = run_screening_pipeline(
        resume          = candidate["resume"],
        job_description = job_description,
        candidate_name  = candidate["name"],
        label           = candidate["label"]
    )
    all_results.append(result)

print("\n" + "="*60)
print("🎉 All 3 candidates processed successfully!")
print("📡 Check LangSmith dashboard for traces:")
print("   https://smith.langchain.com")
print("="*60)

## 📊 Step 8: Display Results — Fit Scores & Explanations

In [ ]:
print("\n" + "="*60)
print("📊 FINAL SCREENING RESULTS")
print("="*60)

for result in all_results:
    name   = result["candidate_name"]
    label  = result["label"]
    sr     = result["score_result"]
    mr     = result["match_result"]
    ei     = result["extracted_info"]

    print(f"\n👤 Candidate  : {name} ({label})")
    print(f"   Experience  : {ei.get('experience_years', 'N/A')}")
    print(f"   Fit Score   : {sr.get('fit_score', 'N/A')}/100")
    print(f"   Grade       : {sr.get('grade', 'N/A')}")
    print(f"   Decision    : {sr.get('recommendation', 'N/A')}")
    print(f"   ✅ Matched  : {', '.join(mr.get('matched_skills', [])) or 'None'}")
    print(f"   ❌ Missing  : {', '.join(mr.get('missing_skills', [])) or 'None'}")
    print(f"   💬 Reason   : {sr.get('explanation', 'N/A')}")
    print(f"   {'─'*50}")

## 🏆 Step 9: Ranking Summary

In [ ]:
# Sort candidates by fit score (descending)
ranked = sorted(
    all_results,
    key=lambda x: x["score_result"].get("fit_score", 0),
    reverse=True
)

print("\n" + "="*60)
print("🏆 CANDIDATE RANKING (Best → Worst Fit)")
print("="*60)
print(f"{'Rank':<6} {'Name':<20} {'Label':<10} {'Score':>7} {'Grade':<12} {'Recommendation'}")
print("-"*75)

for i, result in enumerate(ranked, 1):
    sr = result["score_result"]
    print(
        f"#{i:<5} "
        f"{result['candidate_name']:<20} "
        f"{result['label']:<10} "
        f"{str(sr.get('fit_score', 'N/A')):>5}/100  "
        f"{sr.get('grade', 'N/A'):<12} "
        f"{sr.get('recommendation', 'N/A')}"
    )

print("="*60)

## 🐛 Step 10: LangSmith Debugging — Demonstrating an Incorrect Output
The assignment requires showing **at least one incorrect output** for debugging.

In [ ]:
# ─────────────────────────────────────────────────────────
# Intentional Debug Scenario:
# Submitting an EMPTY/INCOMPLETE resume to observe
# how the pipeline behaves and what LangSmith captures.
# ─────────────────────────────────────────────────────────

incomplete_resume = """
Name: Test Candidate
I am interested in data. I use computers.
"""

print("🐛 DEBUG RUN: Submitting an incomplete/vague resume...")
print("   Purpose: To show LangSmith tracing of an edge case / incorrect output")

debug_result = run_screening_pipeline(
    resume          = incomplete_resume,
    job_description = job_description,
    candidate_name  = "Debug Candidate",
    label           = "Debug/Incorrect"
)

print("\n📋 Debug Result:")
print(f"   Score: {debug_result['score_result'].get('fit_score', 'N/A')}/100")
print(f"   Explanation: {debug_result['score_result'].get('explanation', 'N/A')}")
print("\n⚠️  This run is visible in LangSmith for debugging.")
print("   Check: https://smith.langchain.com → Project: AI-Resume-Screening")

## 🌟 Bonus: Structured JSON Output & Few-Shot Prompting

In [ ]:
# ─────────────────────────────────────────────────────────────
# BONUS: Few-Shot Prompt Example
# Demonstrates how to provide examples to the LLM for better
# structured output and grounding.
# ─────────────────────────────────────────────────────────────

few_shot_scoring_prompt = PromptTemplate(
    input_variables=["match_result", "job_description"],
    template="""
You are a senior hiring manager. Assign a Fit Score (0-100) for the candidate.

--- FEW-SHOT EXAMPLES ---

Example 1:
Matched Skills: Python, Scikit-learn, SQL, TensorFlow, NLP, AWS
Missing Skills: None
Experience Match: true
→ Score: 92, Grade: Excellent, Recommendation: Strongly Recommend

Example 2:
Matched Skills: Python, Pandas, SQL
Missing Skills: TensorFlow, NLP, MLOps, Cloud
Experience Match: false
→ Score: 48, Grade: Average, Recommendation: Consider

Example 3:
Matched Skills: None
Missing Skills: Python, ML, SQL, TensorFlow, NLP, Cloud, MLOps
Experience Match: false
→ Score: 10, Grade: Poor, Recommendation: Do Not Recommend

--- NOW SCORE THIS CANDIDATE ---

Match Analysis:
{match_result}

Job Description:
{job_description}

Output ONLY valid JSON with keys: fit_score, grade, explanation, recommendation.
"""
)

# Create a few-shot scoring chain
few_shot_chain = few_shot_scoring_prompt | llm | output_parser

# Test on the strong candidate
print("🌟 BONUS: Running Few-Shot scoring on Strong candidate...")
strong_result   = all_results[0]
few_shot_output = few_shot_chain.invoke({
    "match_result":    json.dumps(strong_result["match_result"], indent=2),
    "job_description": job_description
})
few_shot_parsed = safe_parse_json(few_shot_output)
print(f"✅ Few-Shot Score: {few_shot_parsed.get('fit_score')}/100")
print(f"   Grade         : {few_shot_parsed.get('grade')}")
print(f"   Explanation   : {few_shot_parsed.get('explanation')}")

## 🏁 Step 11: Export Full Results to JSON

In [ ]:
import json

# Save all results to a JSON file
output_file = "screening_results.json"

with open(output_file, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"✅ Results saved to: {output_file}")
print("   You can include this file in your GitHub repository.")

# Print final summary
print("\n" + "="*60)
print("📌 SUBMISSION CHECKLIST")
print("="*60)
print("✅ LangChain pipeline with LCEL and .invoke()")
print("✅ PromptTemplates for extraction, matching, scoring")
print("✅ 3 candidates screened (Strong, Average, Weak)")
print("✅ LangSmith tracing enabled (LANGCHAIN_TRACING_V2=true)")
print("✅ Debug run with incorrect/incomplete resume")
print("✅ Explainability: score reasoning provided")
print("✅ Bonus: Few-shot prompting implemented")
print("✅ JSON output saved")
print("="*60)
print("🚀 Ready for GitHub push and submission!")